# 32 — Resume Section Detection
**Goal:** Identify section boundaries in unstructured resume text.

A raw resume is a flat wall of lines; its meaning lives in its *structure* — where the summary ends and the experience section begins. This chapter builds that structure: a curated list of known section names, a regex scanner that flags section headers line by line, and a zero-shot ML fallback for layouts that regex cannot read.

**Why it matters for resumes / ATS:** every later extraction step needs to know *which lines belong to which section*. Detecting "EDUCATION", "education", or "Academic Qualifications" as the same section is what lets Ch. 33–38 scope their parsers to the right content block instead of scanning the whole document — and it stops a stray "Skills" mention inside a bullet from being misread as a section header.

## 1. Common Resume Sections

Resumes converge on a small set of standard sections — summary, experience, education, skills, projects — but the *labels* vary wildly: "Work History", "Employment", and "Professional Experience" all mean the same thing. The first job is to enumerate the aliases so a detector can map any of them onto one canonical section.

**What the code does:** builds `SECTIONS`, a flat list of 17 known section names, grouped by meaning — three names for experience, two for education, three for skills. Running it prints `Known sections: 17` and then every entry in the list.

**Why it matters:** the list *is* the detector's vocabulary. A candidate who writes "Core Competencies" instead of "Skills" is only caught because `core competencies` is in the list — so growing this list (a job-title-aware alias table) directly improves recall.

In [ ]:
SECTIONS = [
    "summary", "objective", "profile",
    "experience", "work history", "employment",
    "education", "academic",
    "skills", "technical skills", "core competencies",
    "projects", "certifications", "publications",
    "languages", "interests", "references",
]
print(f"Known sections: {len(SECTIONS)}")
for s in SECTIONS: print(f"  - {s}")

## 2. Regex-Based Section Detection

Section headers are short, standalone lines that *start with* a known section name — an ideal regex target. The recipe: compile one anchored, case-insensitive pattern per section name, then scan the resume line by line.

**What the code does:** `SECTION_PATTERNS` pre-compiles `^<name>` patterns with `re.escape`; `detect_sections()` walks the lines, skips blanks, and when a line matches a pattern **and is shorter than 40 characters** it records `(section, line_index, header_text)`.
- The `< 40` check is the header heuristic: a long line that merely *contains* "experience" ("I gained experience in…") is not a header.
- On the sample resume the run detects 4 sections: `SUMMARY` at line 0, `EXPERIENCE` at line 3, `EDUCATION` at line 7, `SKILLS` at line 10 — exactly the boundaries a human reader would draw.

**Try it:** `re.escape(s)` is what keeps punctuation-heavy names like "C#" or "R&D" safe inside the pattern.

In [ ]:
import re

SECTION_PATTERNS = {s: re.compile(r"^" + re.escape(s), re.IGNORECASE) for s in SECTIONS}

def detect_sections(text):
    lines = text.split("\n")
    sections = []
    for i, line in enumerate(lines):
        line_stripped = line.strip()
        if not line_stripped: continue
        for section_name, pattern in SECTION_PATTERNS.items():
            if pattern.search(line_stripped):
                # Check it looks like a header (short, possibly all-caps)
                if len(line_stripped) < 40:
                    sections.append((section_name, i, line_stripped))
                    break
    return sections

resume = """SUMMARY
Data scientist with Python and ML experience.

EXPERIENCE
Google — Senior Data Scientist, 2020-Present
Built ML pipelines.

EDUCATION
M.S. Computer Science, Stanford University

SKILLS
Python, TensorFlow, PyTorch, SQL, AWS
"""
for name, idx, header in detect_sections(resume):
    print(f"  Line {idx}: [{name:15s}] '{header}'")

## 3. ML-Based Section Classification

Real resumes break the regex contract — "Professional Experience", "Technical Skills & Expertise" — or bury headers in unusual layouts. For those, a zero-shot classifier labels a line *semantically*: it asks a pretrained model (`facebook/bart-large-mnli`) how well each candidate section name matches the line, with no fine-tuning.

**What the code does:** builds a zero-shot `pipeline`, scores each test line against all 17 `SECTIONS`, and prints the best label with its probability. The construction and scoring loop sit inside a `try/except`, so a model that fails to load degrades to a printed fallback message instead of crashing.

**Expected behavior:** each line is labeled with the section it scores highest against, along with that score. Note two caveats: the first run downloads the model (needs network, ~1.6 GB), and the fallback only guards the pipeline call — a hard failure at the `from transformers import pipeline` import itself would still raise. Regex stays the fast, offline default; ML is the escape hatch for hard layouts.

In [ ]:
# Use zero-shot classification for section detection
from transformers import pipeline
try:
    classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
    lines = ["Professional Summary", "Work Experience", "Education", "Skills & Expertise"]
    for line in lines:
        result = classifier(line, SECTIONS)
        print(f"  '{line:25s}' -> {result['labels'][0]:15s} ({result['scores'][0]:.2f})")
except:
    print("Transformers not available. Regex approach works fine for most resumes.")

## 4. Section Content Extraction

Detecting headers is only half the job — the pipeline also needs the *content* that belongs under each header, so the education parser does not accidentally read experience bullets.

**What the code does:** `extract_section_content()` runs the same header check inside a small state machine: any header line flips `in_section` off (a new section just ended the old one), and if that header is the target section it flips `in_section` on. Every following non-blank, non-header line is appended until the next header.

**Verified on the sample resume:** requesting `"experience"` returns exactly the two lines under `EXPERIENCE` — `['Google — Senior Data Scientist, 2020-Present', 'Built ML pipelines.']` — and stops at `EDUCATION`.

**Try it:** call it with `"skills"` and the same state machine walks to the `SKILLS` header and collects its single line — the mechanism every section-scoped parser in Ch. 35–38 relies on.

In [ ]:
def extract_section_content(text, section_name):
    """Extract all content under a detected section."""
    lines = text.split("\n")
    in_section = False
    content = []
    for line in lines:
        ls = line.strip()
        if not ls: continue
        # Check if this line is a section header
        is_header = False
        for sn, pat in SECTION_PATTERNS.items():
            if pat.search(ls) and len(ls) < 40:
                if in_section:
                    in_section = False  # found next section
                if sn == section_name.lower():
                    in_section = True
                break
        else:
            if in_section and ls:
                content.append(ls)
    return content

print("Content under 'experience':")
print(extract_section_content(resume, "experience"))

## Summary: Regex + heuristics for basic detection. Zero-shot ML for complex layouts.

**Section headers are the skeleton of a resume — detect them first, and everything else gets easier.**

Header detection is deliberately cheap: anchored regexes plus a line-length heuristic catch the vast majority of real resumes in microseconds with zero dependencies, and the zero-shot classifier is the safety net for unconventional layouts at the cost of a model download. What matters is that both paths produce the same contract — `(section, line_index, header_text)` — so downstream parsers never care *how* the section was found.

This is the entry point of the extraction pipeline: Ch. 33 starts consuming these sections to pull out skills, and Ch. 35–38 scope their parsers to the content blocks this chapter isolates.